In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName('window practice')\
        .master('local[*]')\
        .getOrCreate()

data = [
    (1, "A", "2024-01-01", 200),
    (2, "A", "2024-01-02", 300),
    (3, "A", "2024-01-03", 250),
    (4, "B", "2024-01-01", 400),
    (5, "B", "2024-01-03", 100),
    (6, "C", "2024-01-02", 500),
    (7, "C", "2024-01-03", 700)
]

columns = ["order_id", "customer_id", "order_date", "amount"]

orders_df = spark.createDataFrame(data, columns)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/28 17:21:11 WARN Utils: Your hostname, Ameys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 100.76.215.140 instead (on interface en0)
26/03/28 17:21:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/28 17:21:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### For each customer_id, assign a row number ordered by order_date.

In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# columns = ["order_id", "customer_id", "order_date", "amount"]

orders_df.withColumn(
    'rn',
    row_number().over(Window.partitionBy(col('customer_id')).orderBy(col('order_date'))
)).show()


[Stage 0:>                                                          (0 + 8) / 8]

+--------+-----------+----------+------+---+
|order_id|customer_id|order_date|amount| rn|
+--------+-----------+----------+------+---+
|       1|          A|2024-01-01|   200|  1|
|       2|          A|2024-01-02|   300|  2|
|       3|          A|2024-01-03|   250|  3|
|       4|          B|2024-01-01|   400|  1|
|       5|          B|2024-01-03|   100|  2|
|       6|          C|2024-01-02|   500|  1|
|       7|          C|2024-01-03|   700|  2|
+--------+-----------+----------+------+---+



### Return only the latest order per customer

In [19]:
orders_df.withColumn(
    'rn',
    row_number().over(Window.partitionBy(col('customer_id')).orderBy(col('order_date').desc()))
).filter(
    col('rn')==1
).show()

+--------+-----------+----------+------+---+
|order_id|customer_id|order_date|amount| rn|
+--------+-----------+----------+------+---+
|       3|          A|2024-01-03|   250|  1|
|       5|          B|2024-01-03|   100|  1|
|       7|          C|2024-01-03|   700|  1|
+--------+-----------+----------+------+---+



### Compute cumulative sum of amount ordered by order_date

In [21]:
orders_df.withColumn(
    'total_amount',
    sum('amount').over(Window.orderBy(col('order_date').asc()))
).show()

26/03/28 17:37:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/28 17:37:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/28 17:37:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/28 17:37:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/28 17:37:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+------+------------+
|order_id|customer_id|order_date|amount|total_amount|
+--------+-----------+----------+------+------------+
|       1|          A|2024-01-01|   200|         600|
|       4|          B|2024-01-01|   400|         600|
|       2|          A|2024-01-02|   300|        1400|
|       6|          C|2024-01-02|   500|        1400|
|       3|          A|2024-01-03|   250|        2450|
|       5|          B|2024-01-03|   100|        2450|
|       7|          C|2024-01-03|   700|        2450|
+--------+-----------+----------+------+------------+



### Get the previous order’s amount

In [34]:
orders_df.withColumn(
    'prev_order',
    lag(col('amount'),1,0).over(Window.partitionBy(col('customer_id')).orderBy(col('order_date')))
).show()

+--------+-----------+----------+------+----------+
|order_id|customer_id|order_date|amount|prev_order|
+--------+-----------+----------+------+----------+
|       1|          A|2024-01-01|   200|         0|
|       2|          A|2024-01-02|   300|       200|
|       3|          A|2024-01-03|   250|       300|
|       4|          B|2024-01-01|   400|         0|
|       5|          B|2024-01-03|   100|       400|
|       6|          C|2024-01-02|   500|         0|
|       7|          C|2024-01-03|   700|       500|
+--------+-----------+----------+------+----------+



### Get the next order’s amount

In [31]:
orders_df.withColumn(
    'next_order',
    lead(col('amount'),1).over(Window.partitionBy(col('customer_id')).orderBy(col('order_date')))
).show()

+--------+-----------+----------+------+----------+
|order_id|customer_id|order_date|amount|next_order|
+--------+-----------+----------+------+----------+
|       1|          A|2024-01-01|   200|       300|
|       2|          A|2024-01-02|   300|       250|
|       3|          A|2024-01-03|   250|      NULL|
|       4|          B|2024-01-01|   400|       100|
|       5|          B|2024-01-03|   100|      NULL|
|       6|          C|2024-01-02|   500|       700|
|       7|          C|2024-01-03|   700|      NULL|
+--------+-----------+----------+------+----------+



In [35]:
data = [
    (1, "A", "2024-01-01", 200),
    (2, "A", "2024-01-02", 300),
    (3, "A", "2024-01-03", 300),
    (4, "A", "2024-01-04", 250),
    (5, "B", "2024-01-01", 400),
    (6, "B", "2024-01-02", 400),
    (7, "B", "2024-01-03", 100),
    (8, "C", "2024-01-01", 500),
    (9, "C", "2024-01-02", 700),
    (10, "C", "2024-01-03", 700)
]

columns = ["order_id", "customer_id", "order_date", "amount"]

orders_df = spark.createDataFrame(data, columns)

### Assign row_number per customer ordered by order_date

In [36]:
orders_df.withColumn(
    'rn',
    row_number().over(Window.partitionBy(col('customer_id')).orderBy(col('order_date')))
).show()

+--------+-----------+----------+------+---+
|order_id|customer_id|order_date|amount| rn|
+--------+-----------+----------+------+---+
|       1|          A|2024-01-01|   200|  1|
|       2|          A|2024-01-02|   300|  2|
|       3|          A|2024-01-03|   300|  3|
|       4|          A|2024-01-04|   250|  4|
|       5|          B|2024-01-01|   400|  1|
|       6|          B|2024-01-02|   400|  2|
|       7|          B|2024-01-03|   100|  3|
|       8|          C|2024-01-01|   500|  1|
|       9|          C|2024-01-02|   700|  2|
|      10|          C|2024-01-03|   700|  3|
+--------+-----------+----------+------+---+



### TASK 2 Find the top 2 highest amount orders per customer

In [40]:
# columns = ["order_id", "customer_id", "order_date", "amount"]

orders_df.withColumn(
    'sorted_orders',
    row_number().over(Window.partitionBy(col('customer_id')).orderBy(col('amount').desc()))
).filter(
    col('sorted_orders')<=2
).orderBy(col('order_id')).show()

+--------+-----------+----------+------+-------------+
|order_id|customer_id|order_date|amount|sorted_orders|
+--------+-----------+----------+------+-------------+
|       2|          A|2024-01-02|   300|            1|
|       3|          A|2024-01-03|   300|            2|
|       5|          B|2024-01-01|   400|            1|
|       6|          B|2024-01-02|   400|            2|
|       9|          C|2024-01-02|   700|            1|
|      10|          C|2024-01-03|   700|            2|
+--------+-----------+----------+------+-------------+



### For each customer, compute:

- difference between current order amount and previous order amount

In [43]:
# columns = ["order_id", "customer_id", "order_date", "amount"]

orders_df.withColumn(
    'prev_order',
    lag(col('amount'),1,0).over(Window.partitionBy(col('customer_id')).orderBy(col('order_date')))
).withColumn(
    'order_amt_diff',
    (col('amount') - col('prev_order'))
).show()

+--------+-----------+----------+------+--------------+
|order_id|customer_id|order_date|amount|order_amt_diff|
+--------+-----------+----------+------+--------------+
|       1|          A|2024-01-01|   200|           200|
|       2|          A|2024-01-02|   300|           100|
|       3|          A|2024-01-03|   300|             0|
|       4|          A|2024-01-04|   250|           -50|
|       5|          B|2024-01-01|   400|           400|
|       6|          B|2024-01-02|   400|             0|
|       7|          B|2024-01-03|   100|          -300|
|       8|          C|2024-01-01|   500|           500|
|       9|          C|2024-01-02|   700|           200|
|      10|          C|2024-01-03|   700|             0|
+--------+-----------+----------+------+--------------+



### For each customer:

- find all orders where amount is greater than previous order

In [46]:
orders_df.withColumn(
    'prev_orders',
    lag(col('amount'),1,0).over(Window.partitionBy(col('customer_id')).orderBy(col('amount')))
).filter(
    col('amount')<col('prev_orders')
).show()

+--------+-----------+----------+------+-----------+
|order_id|customer_id|order_date|amount|prev_orders|
+--------+-----------+----------+------+-----------+
|       3|          A|2024-01-03|   300|        300|
|       6|          B|2024-01-02|   400|        400|
|      10|          C|2024-01-03|   700|        700|
+--------+-----------+----------+------+-----------+



### For each customer:

- compute a running average of amount

In [47]:
orders_df.withColumn(
    'running_sum',
    sum(col('amount')).over(Window.partitionBy(col('customer_id')).orderBy(col('order_date')))
).show()

+--------+-----------+----------+------+-----------+
|order_id|customer_id|order_date|amount|running_sum|
+--------+-----------+----------+------+-----------+
|       1|          A|2024-01-01|   200|        200|
|       2|          A|2024-01-02|   300|        500|
|       3|          A|2024-01-03|   300|        800|
|       4|          A|2024-01-04|   250|       1050|
|       5|          B|2024-01-01|   400|        400|
|       6|          B|2024-01-02|   400|        800|
|       7|          B|2024-01-03|   100|        900|
|       8|          C|2024-01-01|   500|        500|
|       9|          C|2024-01-02|   700|       1200|
|      10|          C|2024-01-03|   700|       1900|
+--------+-----------+----------+------+-----------+



### Find the second highest order amount per customer

Important: handle duplicates correctly

### For each customer:

- mark rows as:

- "increase" if amount > previous
- "decrease" if amount < previous
- "same" if equal

In [59]:
orders_df.withColumn(
    'prev_order',
    lag(col('amount'),1).over(Window.partitionBy(col('customer_id')).orderBy(col('order_date')))
).withColumn(
    'status',
    when(col('amount')<col('prev_order'),'decrease')
    .when(col('amount')>col('prev_order'),'increase').otherwise('equal')
).show()

+--------+-----------+----------+------+----------+--------+
|order_id|customer_id|order_date|amount|prev_order|  status|
+--------+-----------+----------+------+----------+--------+
|       1|          A|2024-01-01|   200|      NULL|   equal|
|       2|          A|2024-01-02|   300|       200|increase|
|       3|          A|2024-01-03|   300|       300|   equal|
|       4|          A|2024-01-04|   250|       300|decrease|
|       5|          B|2024-01-01|   400|      NULL|   equal|
|       6|          B|2024-01-02|   400|       400|   equal|
|       7|          B|2024-01-03|   100|       400|decrease|
|       8|          C|2024-01-01|   500|      NULL|   equal|
|       9|          C|2024-01-02|   700|       500|increase|
|      10|          C|2024-01-03|   700|       700|   equal|
+--------+-----------+----------+------+----------+--------+



In [55]:
new.withColumn(
    'status',
    when(
        col('amount') < col('prev_order'),'decrease'
    )
).printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- prev_order: long (nullable = true)
 |-- next_order: long (nullable = true)
 |-- status: string (nullable = true)

